# Run the Stage 1 GHI pipeline (orchestrator)

[!] writing to `*_v2` tables so his originals are never touched. [!] 

```
build_structured_data  →  build_split_days  →  build_ghi_model  →  build_all_uncurtailedpv
        (_v2)                   (_v2)                (_v2)                  (_v2)
```
**Cost note:** the structured_data build scans `ts` (billions of rows). Run the
**tiny test slice first** (one month, one site-part) and confirm it works before
scaling to the full year. Athena bills by data scanned.

In [ ]:
# Bootstrap: paths + imports
import sys, pathlib
SHARED = pathlib.Path(r"../../shared")
STAGE1  = pathlib.Path(r"../stage1_ghi_pipeline")
sys.path.insert(0, str(SHARED))
sys.path.insert(0, str(STAGE1))

from aws_config import aq            # your existing Athena helper
from ciccada_config import SAI       # 'solar_analytics_iceberg'

import build_structured_data   as b1
import build_split_days        as b2
import build_ghi_model         as b3
import build_mape_quality_gate as b3b
import build_all_uncurtailedpv as b4

print("Target tables (note the _v2 suffix. Original results untouched):")
print(" ", b1.TARGET)
print(" ", b2.TARGET)
print(" ", b3.TARGET)
print(" ", b4.TARGET)

## Step 1. Structured_data_v2

In [ ]:
# 1a. Create the empty table (safe: drops & recreates only the _v2 table)
print(b1.create_table(aq, database=SAI))

In [ ]:
# 1b. TEST SLICE FIRST. 
# One month, one of 8 site-parts.
# Confirm this completes and validate() looks sane BEFORE the full run.
b1.run_slice(aq, database=SAI, year=2024, months=[1], n_parts=8, parts=[0])

In [ ]:
# 1c. Validate the test slice
b1.validate(aq, database=SAI)

In [ ]:
# 1d. FULL RUN. all months, all 8 site-parts, for the year(s) dedfined in the pipeline config.
#     Only run this once the test slice looks right. This is the longer part.
#     (Re-running create_table first would wipe the test slice
#      that's expected so the full run below reloads everything cleanly.)
print(b1.create_table(aq, database=SAI))
# Both years, 2024 and 2025:
b1.run_slice(aq, database=SAI, year=2024, months=range(1, 13), n_parts=16)
b1.run_slice(aq, database=SAI, year=2025, months=range(1, 13), n_parts=16)
b1.validate(aq, database=SAI)

## Step 2. split_days_v2

In [ ]:
print(b2.create_table(aq, database=SAI))
print(b2.run(aq, database=SAI))
b2.validate(aq, database=SAI)

## Step 3. pv_ghi_norm_model_v2

In [ ]:
print(b3.create_table(aq, database=SAI))
b3.run(aq, database=SAI, years=(2024, 2025))   # ONE call, both years
b3.validate(aq, database=SAI)

In [ ]:
# Step 3b. MAPE quality gate, now saving an auditable CSV
MAPE_CSV = "mape_under50_sites.csv"
mape_df, good_sites = b3b.run(aq, database=SAI, csv_path=MAPE_CSV)

## Step 4. all_uncurtailedpv_v2

In [ ]:
MAPE_CSV = r"mape_under50_sites.csv" 
print(b4.create_table(aq, database=SAI))
b4.run_year(aq, database=SAI, year=2024, mape_csv_path=MAPE_CSV, n_parts=6)
b4.run_year(aq, database=SAI, year=2025, mape_csv_path=MAPE_CSV, n_parts=6)
b4.validate(aq, database=SAI)

## Done. Stage 1 rebuilt

# Optional comparisons with original

## Optional 1: Report on validation metrics

In [ ]:
# structured data built
b1.validate(aq, database=SAI)

In [ ]:
# build_split_days
b2.validate(aq, database=SAI)

In [ ]:
# GHI model
b3.validate(aq, database=SAI)

In [ ]:
# Uncurtailed PV
b4.validate(aq, database=SAI)

In [ ]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, t_stamp
        FROM all_uncurtailedpv_v2
        GROUP BY site_id, t_stamp
        HAVING count(*) > 1
    )
""", database=SAI)

In [ ]:
aq("""
    SELECT count(*) AS n_sites_with_multiple_caps
    FROM (
        SELECT site_id, count(DISTINCT ac_capacity_kw) AS n
        FROM meta_up23c
        WHERE is_pv = True
        GROUP BY site_id
        HAVING count(DISTINCT ac_capacity_kw) > 1
    )
""", database=SAI)

In [ ]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, tod_bin
        FROM pv_ghi_norm_model_v2
        GROUP BY site_id, tod_bin
        HAVING count(*) > 1
    )
""", database=SAI)

## Optional 2: Comapre with V1 (original results included on the Milestone 3 report)

In [ ]:
# Comparison: _v2 vs originals
compare = aq(f"""
    SELECT 
        'original' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(min(V), 1) AS v_min,
        round(avg(V), 1) AS v_avg,
        round(max(V), 1) AS v_max,
        round(avg(P_kw_norm), 4) AS p_norm_avg
    FROM structured_data
    UNION ALL
    SELECT
        'v2' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(min(V), 1) AS v_min,
        round(avg(V), 1) AS v_avg,
        round(max(V), 1) AS v_max,
        round(avg(P_kw_norm), 4) AS p_norm_avg
    FROM structured_data_v2
""", database=SAI)
print("structured_data: original vs v2")
print(compare.to_string(index=False))




In [ ]:
#  What to expect: 
# Site counts should be similar but not identica
# _v2 should have slightly fewer sites because of [flex_export exclusion, ~539 sites removed]. 
# The v_max should be higher in _v2 because of the switch from avg to max voltage. 
# The max_uncurt should be lower in _v2 because now it caps at nameplate.

compare_unc = aq(f"""
    SELECT
        'original' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(avg(uncurtailed_P), 2) AS avg_uncurt,
        round(max(uncurtailed_P), 2) AS max_uncurt
    FROM all_uncurtailedpv
    UNION ALL
    SELECT
        'v2' AS version,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        round(avg(uncurtailed_P), 2) AS avg_uncurt,
        round(max(uncurtailed_P), 2) AS max_uncurt
    FROM all_uncurtailedpv_v2
""", database=SAI)
print("\nall_uncurtailedpv: original vs v2")
print(compare_unc.to_string(index=False))

In [ ]:
funnel = aq(f"""
    SELECT 'structured_data_v2' AS stage, count(DISTINCT site_id) AS n_sites
    FROM structured_data_v2
    UNION ALL
    SELECT 'split_days_v2', count(DISTINCT site_id) FROM split_days_v2
    UNION ALL
    SELECT 'ghi_model_v2', count(DISTINCT site_id) FROM pv_ghi_norm_model_v2
    UNION ALL
    SELECT 'mape_csv', count(DISTINCT site_id) FROM all_uncurtailedpv_v2
""", database=SAI)
print("\nSite funnel through Stage 1:")
print(funnel.to_string(index=False))

In [ ]:
coverage = aq(f"""
    SELECT year, month, count(*) AS n_rows, count(DISTINCT site_id) AS n_sites
    FROM structured_data_v2
    GROUP BY year, month
    ORDER BY year, month
""", database=SAI)
print("\nMonthly coverage:")
print(coverage.to_string(index=False))

In [ ]:
# What to expect: _v2 should show higher average voltage and a higher percentage above 240V. 
# Vmax catches the high phase that avg was hiding. 
# The difference quantifies how much conformance was being undercounted.


r1_impact = aq(f"""
    SELECT
        round(avg(V), 2) AS v2_avg_of_max,
        round(count(CASE WHEN V > 240 THEN 1 END) * 100.0 / count(*), 2)
            AS pct_above_240_v2
    FROM structured_data_v2
    WHERE P_kw_norm > 0.05
""", database=SAI)

r1_original = aq(f"""
    SELECT
        round(avg(V), 2) AS orig_avg_of_avg,
        round(count(CASE WHEN V > 240 THEN 1 END) * 100.0 / count(*), 2)
            AS pct_above_240_orig
    FROM structured_data
    WHERE P_kw_norm > 0.05
""", database=SAI)

print("\nFix impact (avg→max voltage):")
print(f"  Original avg(voltage):  mean={r1_original['orig_avg_of_avg'].iloc[0]}V, "
      f"{r1_original['pct_above_240_orig'].iloc[0]}% above 240V")
print(f"  v2 max(voltage):        mean={r1_impact['v2_avg_of_max'].iloc[0]}V, "
      f"{r1_impact['pct_above_240_v2'].iloc[0]}% above 240V")

In [ ]:
r2_impact = aq(f"""
    SELECT
        count(DISTINCT o.site_id) AS in_original_only
    FROM (SELECT DISTINCT site_id FROM structured_data) o
    LEFT JOIN (SELECT DISTINCT site_id FROM structured_data_v2) v
        ON o.site_id = v.site_id
    WHERE v.site_id IS NULL
""", database=SAI)
print(f"\n[Flex export] impact: {int(r2_impact['in_original_only'].iloc[0])} sites in original "
      f"but excluded from v2 (flex_export + other filter differences)")

In [ ]:
by_state = aq(f"""
    SELECT m.state, count(DISTINCT sd.site_id) AS n_sites
    FROM structured_data_v2 sd
    JOIN (SELECT DISTINCT site_id, state FROM meta_up23c) m ON sd.site_id = m.site_id
    GROUP BY m.state
    ORDER BY n_sites DESC
""", database=SAI)
print("\nSites by state:")
print(by_state.to_string(index=False))